# 실습 C. 테스트를 다 통과한 AI 패치 판단하기 (약 30분)

> **이 실습에서 배울 것: 테스트를 전부 통과한 AI 패치에도 버그가 있을 수 있습니다.** 테스트는 사람이 만든 상황만 확인합니다. Dafny 검증은 **모든 상태**를, 차분 테스트는 **무작위로 만든 많은 상황**을 확인해서, 테스트가 만들지 못한 상황의 버그를 찾습니다.

**상황 (변경 요청 1, 상품기획팀).** "저녁에 이벤트가 몰리면 대기열(8칸)이 가득 차서 이벤트가 거부됩니다. 평소에는 8칸만 쓰고, 필요할 때만 최대 32칸까지 늘려 주세요."

AI에게 이 요청을 줬더니 패치 두 개가 나왔습니다. **둘 다 테스트를 전부 통과합니다.** 어느 것을 내보내도 될까요?

| | 대기열이 가득 차면 |
|---|---|
| 패치 A | 배열을 2배로 늘림. 이벤트를 **배열에 있는 자리 그대로** 새 배열에 복사하고, 맨 앞 위치(`head`)는 그대로 둠 |
| 패치 B | 배열을 2배로 늘림. 이벤트를 **오래된 것부터 순서대로** 새 배열 앞쪽에 옮기고, `head = 0` |

In [ ]:
import lab   # 실습 도우미: lab.run(...)이 make 명령을 실행하고 결과 보고서를 아래에 보여 줍니다

## 두 패치가 다른 곳: `grow()`

배열을 늘리는 메서드 `grow()`만 다릅니다.

In [1]:
lab.side_by_side([
    (("scenarios/req1/ai-A/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", 48, 53),
     ("scenarios/req1/ai-B/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", 48, 56),
     "A는 배열을 0번 칸부터 <b>있는 그대로</b> 복사하고 <code>head</code>는 그대로. B는 <code>head</code>부터 <b>오래된 순서대로</b> 0번 칸부터 옮기고 <code>head = 0</code>"),
], title="패치 A의 grow() ↔ 패치 B의 grow()", headers=("패치 A", "패치 B", "차이"))

어느 쪽이 맞는지 테스트 → Dafny → 차분 테스트 순서로 확인합니다.

## C-1. 테스트만으로 판단하기 (5분)

In [2]:
lab.run("req1-tests")

$ make req1-tests


ai-A   tests: 40, passed: 40, failed: 0


ai-B   tests: 42, passed: 42, failed: 0


패치 A 40/40, 패치 B 42/42. **테스트만 보면 둘 다 내보낼 수 있습니다.** 개수가 다른 것은 각 패치가 새로 더한 테스트 수가 달라서입니다. 레거시 테스트 38개에 패치 A는 2개를, 패치 B는 4개를 더했습니다.

두 패치 모두 "대기열이 32칸까지 늘어나는지" 테스트를 더했지만, 모두 **빈 대기열**에 이벤트를 넣기만 하면서 늘립니다. 앞쪽 이벤트를 몇 개 꺼내서 대기열이 배열 끝을 넘어 한 바퀴 돈 상태(`head`가 0이 아님)에서 늘리는 테스트는 없습니다.

## C-2. Dafny로 판단하기 (10분)

각 패치의 대기열을 그대로 옮긴 Dafny 모델(`scenarios/req1/model/`)을 검증합니다. 계약은 레거시 대기열 모델(`formal/RingQueue.dfy`)과 같고, 상한만 8 → 32입니다. 계약에는 "배열을 `head`부터 읽으면 대기열 내용과 같다"(`Valid`)가 들어 있습니다. Dafny는 이것을 **모든 상태**에서 지키는지 증명합니다.

In [3]:
lab.run("req1-verify")

$ make req1-verify
Dafny Types.dfy RingQueue_A.dfy                검증 24개, 오류 1개
Dafny Types.dfy RingQueue.dfy Dispatcher.dfy P 검증 73개, 오류 0개


**패치 A: `Grow`에서 오류.** 보고서는 이렇게 읽습니다.

- **빨간 줄 112줄**: `Grow`가 끝나는 곳. 여기서 약속을 지키지 못했습니다.
- **노란 줄 111줄 `Valid()`**: 지키지 못한 약속은 `Grow`의 사후조건 `ensures Valid()`입니다.
- **노란 줄 32줄 `Inv()`**: `Valid()` 안에서도 `Inv()` 부분입니다. `Inv()`는 "배열과 대기열 내용(`elems`)이 서로 맞는다"는 약속이고, 그 핵심이 아래 한 줄입니다.

In [4]:
lab.annotate("scenarios/req1/model/RingQueue_A.dfy", [
    (44, 44, "inv", "<b>i번째 내용 = 배열의 (head + i)번째 칸</b>(끝을 넘으면 앞으로 돌아감). <b>패치 A의 Grow 뒤에 이것이 깨집니다</b>. <code>Wrap(k)</code>는 이 모델 안에서 정의한 함수로 Java 코드의 <code>k % buf.length</code> 계산을 나타냄, <code>as int</code>는 32비트 정수를 Dafny 정수로 쓴다는 뜻"),
], start=44, end=44, title="RingQueue_A.dfy: Inv()의 핵심 조건")

이 조건이 왜 깨지는지 그림으로 보면 이렇습니다.

```
늘리기 전  (head=2, 내용 c,d,e,f)
  [ e | f | c | d ]                       c,d가 뒤쪽, e,f가 앞쪽에 있음

패치 A    (있는 그대로 복사, head=2 유지)
  [ e | f | c | d | _ | _ | _ | _ ]       head=2부터 읽으면 c,d,_,_ ...  ← e,f가 사라짐

패치 B    (순서대로 옮김, head=0)
  [ c | d | e | f | _ | _ | _ | _ ]       head=0부터 읽으면 c,d,e,f
```

테스트는 빈 대기열에서만 늘려 봤기 때문에 이 상황을 만들지 않았습니다.

**패치 B: 통과.**

## C-3. 차분 테스트로 판단하기 (10분)

> **이 단계의 포인트: Dafny가 검증한 것은 Java가 아니라 Java를 옮긴 모델입니다. 차분 테스트는 실제 Java가 검증된 모델과 같게 동작하는지 봅니다.**

기준으로 쓰는 모델은 **패치 B의 모델**(`scenarios/req1/model/RingQueue.dfy`)입니다. C-2에서 검증을 통과한, 계약대로 동작한다고 증명된 모델이기 때문입니다. 패치 A의 모델은 검증에 실패했으므로 기준이 될 수 없습니다. 같은 무작위 연산을 각 Java와 이 모델에 넣고, 매 연산 뒤의 결과를 비교합니다. 같으면 그 Java도 모델처럼 믿을 수 있고, 다르면 그 Java가 모델과 다르게 동작하는 연산열이 나옵니다.

| 비교 | 예상 | 뜻 |
|---|---|---|
| 패치 A의 Java vs 패치 B의 모델 | 불일치 | C-2에서 찾은 버그가 실제 Java에도 있음 |
| 패치 B의 Java vs 패치 B의 모델 | 일치 | 패치 B의 Java는 검증된 모델대로 동작함 |
| 레거시 Java vs 패치 B의 모델 | 불일치 | 요청대로 바뀐 동작(상한 증가) |

In [5]:
lab.run("req1-difftest")

$ make req1-difftest


[queue] MISMATCH (run 0, seed 42)


[queue] 300 runs, 18000 ops: Java and model agree


[dispatcher] 300 runs, 18300 ops: Java and model agree


[power] 300 runs, 18000 ops: Java and model agree


[queue] MISMATCH (run 0, seed 42)


- **패치 A**: 연산 11개 뒤 마지막 단계에서 이벤트 14가 사라지고 빈 칸(null)이 생겼습니다. 테스트가 만들지 않은 "한 바퀴 돈 상태에서 늘리기"를 무작위 연산이 만들어 냈습니다.
- **레거시**: 5번째 `offer`에서 Java는 `FULL`, 모델은 `OK`입니다. 차분 테스트는 대기열을 4칸으로 만드는데, 레거시는 4칸이 끝이고 모델은 늘어나기 때문입니다. 버그가 아니라 **이번 요청으로 바뀐 동작**입니다. 차분 테스트는 "다르다"까지만 알려 줍니다. 그 차이가 요청한 변화인지는 사람이 요청서와 맞춰 봐야 합니다.

## 정리

| | 테스트 | Dafny | 차분 테스트 | 판단 |
|---|---|---|---|---|
| 패치 A | 통과 | `Grow` 실패 | 불일치 (이벤트가 사라짐) | 버그. 내보낼 수 없음 |
| 패치 B | 통과 | 통과 | 일치 | 내보낼 수 있음 |

**테스트를 더 잘 쓰면 되지 않을까요?** 이 버그는 "한 바퀴 돈 상태"와 "늘리기"가 겹칠 때만 납니다. 이런 조합을 사람이 미리 다 떠올리기는 어렵습니다. Dafny의 계약은 **모든 상태**를 다루고, 차분 테스트는 조합을 **무작위로 많이** 만들어 냅니다.